In [1]:
!apt-get update -qq

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [5]:

!apt-get install -y -qq libatk-1.0-0 libatk-bridge2.0-0 libcups2 libxkbcommon0 libxcomposite1 libxdamage1 libxrandr2 libgbm1 libpangocairo-1.0-0 libpango-1.0-0 libcairo2 libasound2 libnss3 libxshmfence1

E: Unable to locate package libatk-1.0-0
E: Couldn't find any package by glob 'libatk-1.0-0'
E: Couldn't find any package by regex 'libatk-1.0-0'


In [6]:
!pip -q install -U playwright
!playwright install chromium

In [7]:
!playwright install-deps chromium
!playwright install chromium

Installing dependencies...
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-freefont-ttf is already the newest version (20120503-10build1

In [46]:
# from playwright.async_api import async_playwright

# url = "https://hansard.parliament.nz/hansard-transcript/2026-01-28"

# async def scrape():
#     async with async_playwright() as p:
#         browser = await p.chromium.launch(headless=True)
#         page = await browser.new_page()

#         await page.goto(url, wait_until="domcontentloaded")

#         await page.wait_for_selector("body main", timeout=30_000)
#         await page.wait_for_selector("body main article", timeout=60_000)

#         article = page.locator("body main article").first

#         text = await article.inner_text()
#         html = await article.evaluate("el => el.outerHTML")

#         print("=== TEXT (first 4000 chars) ===")
#         print(text[:40000])

#         # If you want HTML:
#         # print("=== HTML ===")
#         # print(html)

#         await browser.close()

# await scrape()

In [47]:
# !wget https://hansard.parliament.nz/api/resources/transcript/2026-01-28

In [48]:
# import json
# from datetime import datetime
# from playwright.async_api import async_playwright


# def _validate_date(date_str: str) -> str:
#     datetime.strptime(date_str, "%Y-%m-%d")
#     return date_str


# async def get_hansard_article(date_str: str) -> dict:
#     """
#     Fetch transcript for date and return:
#       - text: cleaned text content of the first <article> inside <main>
#       - html: raw HTML of that <article> (outerHTML)
#       - source: selector used
#       - url: API URL used
#     """
#     date_str = _validate_date(date_str)
#     url = f"https://hansard.parliament.nz/api/resources/transcript/{date_str}"

#     async with async_playwright() as p:
#         browser = await p.chromium.launch(headless=True, args=["--no-sandbox"])
#         page = await browser.new_page()

#         await page.goto(url, wait_until="networkidle", timeout=60_000)

#         body_text = (await page.inner_text("body")).strip()

#         # Detect bot-verification page
#         if "Verifying your browser" in body_text or "Incident ID" in body_text:
#             await browser.close()
#             raise RuntimeError("Blocked by anti-bot verification page.")

#         # API often returns JSON-string containing HTML
#         html_doc = None
#         if body_text.startswith('"') and "\\u003C" in body_text:
#             html_doc = json.loads(body_text)
#             await page.set_content(html_doc, wait_until="domcontentloaded")
#         else:
#             # already HTML
#             html_doc = await page.content()

#         # Now parse DOM and extract <main><article>
#         await page.wait_for_selector("body main", timeout=30_000)

#         article = page.locator("body main article").first

#         # outerHTML gives <article> ... </article>
#         article_html = await article.evaluate("el => el.outerHTML")
#         article_text = await article.inner_text()

#         await browser.close()

#     return {
#         "url": url,
#         "source": "body > main > article:first",
#         "text": article_text,
#         "html": article_html,
#     }


In [49]:
# transcript = await get_hansard_transcript("2026-01-28")
# print(transcript)

In [12]:
import json
from datetime import datetime
from pathlib import Path
from playwright.async_api import async_playwright


def _validate_date(date_str: str) -> str:
    datetime.strptime(date_str, "%Y-%m-%d")
    return date_str


async def get_hansard_body(date_str: str, save: bool = True, out_dir: str = ".") -> dict:
    """
    Uses the same approach as your working code:
      1) goto API URL
      2) read body text
      3) json.loads() it into real HTML
      4) page.set_content(real_html)
      5) extract <body> outerHTML + innerText

    By default saves:
      - hansard_<date>_body.txt
      - hansard_<date>_body.html
    """
    date_str = _validate_date(date_str)
    url = f"https://hansard.parliament.nz/api/resources/transcript/{date_str}"

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True, args=["--no-sandbox"])
        page = await browser.new_page()

        await page.goto(url, wait_until="networkidle", timeout=60_000)

        api_page_text = (await page.inner_text("body")).strip()

        # Detect bot-verification page
        if "Verifying your browser" in api_page_text or "Incident ID" in api_page_text:
            await browser.close()
            raise RuntimeError("Blocked by anti-bot verification page.")

        # The API response is typically a JSON string containing the HTML doc
        if api_page_text.startswith('"'):
            html_doc = json.loads(api_page_text)
        else:
            # If it ever returns raw HTML instead of JSON string
            html_doc = await page.content()

        # Load the real transcript HTML into the DOM
        await page.set_content(html_doc, wait_until="domcontentloaded")

        # Extract full <body>...</body>
        await page.wait_for_selector("body", timeout=30_000)
        body = page.locator("body").first

        body_html = await body.evaluate("el => el.outerHTML")
        body_text = await body.inner_text()

        await browser.close()

    saved_txt_path = None
    saved_html_path = None

    if save:
        txt_path = out_dir / f"hansard_{date_str}_body.txt"
        html_path = out_dir / f"hansard_{date_str}_body.html"
        txt_path.write_text(body_text, encoding="utf-8")
        html_path.write_text(body_html, encoding="utf-8")
        saved_txt_path = str(txt_path)
        saved_html_path = str(html_path)

    return {
        "url": url,
        "source": "body",
        "text": body_text,
        "html": body_html,
        "saved_txt_path": saved_txt_path,
        "saved_html_path": saved_html_path,
    }


In [13]:
# data = await get_hansard_body("2026-01-28")  # save=True default
# print(data["saved_txt_path"], data["saved_html_path"])
# print(data["text"][:2000])


In [14]:
# data = await get_hansard_body("2026-01-28")  # save=True default
# print(data["saved_txt_path"], data["saved_html_path"])
# print(data["text"][:2000])


In [15]:
from datetime import datetime, timedelta
from typing import List

def dates_between(start_date: str, end_date: str) -> List[str]:
    """
    Returns list of dates (YYYY-MM-DD) from start_date to end_date inclusive.
    """
    start = datetime.strptime(start_date, "%Y-%m-%d").date()
    end = datetime.strptime(end_date, "%Y-%m-%d").date()

    if start > end:
        start, end = end, start

    out = []
    cur = start
    while cur <= end:
        out.append(cur.isoformat())
        cur += timedelta(days=1)
    return out


In [16]:
import asyncio
from typing import Dict, Any, List

async def scrape_range(start_date: str, end_date: str, out_dir: str = ".", save: bool = True, batch_size: int = 5, max_retries: int = 3):
    """
    Loops through all dates in range and calls get_hansard_body().
    Downloads in batches concurrently using asyncio.gather.
    If a date fails, logs and continues. Retries specific failures.
    Returns:
      - results: list of successful result dicts
      - failures: list of {"date":..., "error":...}
    """
    date_list = dates_between(start_date, end_date)

    results: List[Dict[str, Any]] = []
    initial_failures: List[Dict[str, str]] = []
    retry_queue: List[Dict[str, Any]] = [] # [{'date': 'YYYY-MM-DD', 'attempt': 0}]

    # Initial pass
    for i in range(0, len(date_list), batch_size):
        batch_dates = date_list[i : i + batch_size]
        tasks = []
        for d in batch_dates:
            tasks.append(get_hansard_body(d, save=save, out_dir=out_dir))

        batch_results = await asyncio.gather(*tasks, return_exceptions=True)

        for d, res in zip(batch_dates, batch_results):
            if isinstance(res, Exception):
                error_msg = str(res)
                if "Blocked by anti-bot verification page." in error_msg and max_retries > 0:
                    retry_queue.append({'date': d, 'attempt': 0})
                    print(f"⚠️ {d} BLOCKED, adding to retry queue.")
                else:
                    initial_failures.append({"date": d, "error": error_msg})
                    print(f"⚠️ {d} FAILED -> {error_msg}")
            else:
                results.append(res)
                print(f"✅ {d} OK  -> {res.get('saved_txt_path')}")

        await asyncio.sleep(1) # Polite delay between batches

    # Retry pass
    while retry_queue:
        current_retry_batch = []
        next_retry_queue = []

        # Prepare a batch for retrying
        for item in retry_queue:
            if item['attempt'] < max_retries:
                current_retry_batch.append(item)
            else:
                initial_failures.append({"date": item['date'], "error": f"Failed after {max_retries} retries: Blocked by anti-bot verification page."})

        if not current_retry_batch: # No more items to retry
            break

        print(f"\n🔄 Retrying {len(current_retry_batch)} dates (attempt {current_retry_batch[0]['attempt'] + 1}/{max_retries})...")
        retry_tasks = []
        retry_dates_map = {item['date']: item for item in current_retry_batch}

        for item in current_retry_batch:
            retry_tasks.append(get_hansard_body(item['date'], save=save, out_dir=out_dir))

        retry_batch_results = await asyncio.gather(*retry_tasks, return_exceptions=True)

        for date_item, res in zip(current_retry_batch, retry_batch_results):
            d = date_item['date']
            if isinstance(res, Exception):
                error_msg = str(res)
                if "Blocked by anti-bot verification page." in error_msg:
                    date_item['attempt'] += 1
                    if date_item['attempt'] < max_retries:
                        next_retry_queue.append(date_item)
                        print(f"⚠️ {d} BLOCKED again, re-adding to retry queue (attempt {date_item['attempt'] + 1}).")
                    else:
                        initial_failures.append({"date": d, "error": f"Failed after {max_retries} retries: Blocked by anti-bot verification page."})
                        print(f"❌ {d} FAILED permanently after {max_retries} retries.")
                else:
                    initial_failures.append({"date": d, "error": error_msg})
                    print(f"⚠️ {d} FAILED during retry -> {error_msg}")
            else:
                results.append(res)
                print(f"✅ {d} OK on retry -> {res.get('saved_txt_path')}")

        retry_queue = next_retry_queue
        if retry_queue: # Only sleep if there are still items to retry
            await asyncio.sleep(5) # Longer polite delay between retry batches

    return results, initial_failures

In [ ]:
results, failures = await scrape_range("2012-01-01", "2026-01-28", out_dir="hansard_out", save=True, batch_size=5, max_retries=3)

print("Success:", len(results))
print("Failures:", len(failures))
failures[:5]

Streaming output truncated to the last 5000 lines.
✅ 2012-05-24 OK  -> hansard_out/hansard_2012-05-24_body.txt
✅ 2012-05-25 OK  -> hansard_out/hansard_2012-05-25_body.txt
✅ 2012-05-26 OK  -> hansard_out/hansard_2012-05-26_body.txt
✅ 2012-05-27 OK  -> hansard_out/hansard_2012-05-27_body.txt
✅ 2012-05-28 OK  -> hansard_out/hansard_2012-05-28_body.txt
✅ 2012-05-29 OK  -> hansard_out/hansard_2012-05-29_body.txt
✅ 2012-05-30 OK  -> hansard_out/hansard_2012-05-30_body.txt
✅ 2012-05-31 OK  -> hansard_out/hansard_2012-05-31_body.txt
✅ 2012-06-01 OK  -> hansard_out/hansard_2012-06-01_body.txt
✅ 2012-06-02 OK  -> hansard_out/hansard_2012-06-02_body.txt
⚠️ 2012-06-03 BLOCKED, adding to retry queue.
⚠️ 2012-06-04 BLOCKED, adding to retry queue.
⚠️ 2012-06-05 BLOCKED, adding to retry queue.
✅ 2012-06-06 OK  -> hansard_out/hansard_2012-06-06_body.txt
✅ 2012-06-07 OK  -> hansard_out/hansard_2012-06-07_body.txt
✅ 2012-06-08 OK  -> hansard_out/hansard_2012-06-08_body.txt
✅ 2012-06-09 OK  -> hansard_out

# remove null files

In [38]:
!find "/content/hansard_out" -type f -name "*.txt" -size 4c

/content/hansard_out/hansard_2015-09-03_body.txt
/content/hansard_out/hansard_2018-05-14_body.txt
/content/hansard_out/hansard_2016-11-12_body.txt
/content/hansard_out/hansard_2014-11-19_body.txt
/content/hansard_out/hansard_2018-02-25_body.txt
/content/hansard_out/hansard_2012-12-09_body.txt
/content/hansard_out/hansard_2024-03-12_body.txt
/content/hansard_out/hansard_2022-04-08_body.txt
/content/hansard_out/hansard_2018-08-26_body.txt
/content/hansard_out/hansard_2022-03-25_body.txt
/content/hansard_out/hansard_2021-12-06_body.txt
/content/hansard_out/hansard_2019-06-01_body.txt
/content/hansard_out/hansard_2024-11-28_body.txt
/content/hansard_out/hansard_2021-01-09_body.txt
/content/hansard_out/hansard_2017-03-13_body.txt
/content/hansard_out/hansard_2024-03-02_body.txt
/content/hansard_out/hansard_2020-07-03_body.txt
/content/hansard_out/hansard_2022-01-09_body.txt
/content/hansard_out/hansard_2025-03-23_body.txt
/content/hansard_out/hansard_2018-08-04_body.txt
/content/hansard_out

In [39]:
import os

BASE_DIR = "/content/hansard_out"

deleted = []
remaining = []

for root, _, files in os.walk(BASE_DIR):
    for filename in files:
        if filename.endswith(".txt"):
            txt_path = os.path.join(root, filename)

            # Read and check content
            try:
                with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
                    content = f.read().strip()
            except Exception as e:
                print(f"Failed to read {txt_path}: {e}")
                continue

            # If text content is empty
            if not content or content == "null":
                base_name = filename[:-4]  # remove .txt
                html_path = os.path.join(root, base_name + ".html")

                # Delete .txt
                os.remove(txt_path)
                deleted.append(txt_path)

                # Delete .html if it exists
                if os.path.exists(html_path):
                    os.remove(html_path)
                    deleted.append(html_path)

                print(f"Deleted {filename}")
            else:
                remaining.append(txt_path)

print(f"Deleted {len(deleted)} files:")
print(f"Remaining {len(remaining)} files:")


Deleted hansard_2015-09-03_body.txt
Deleted hansard_2018-05-14_body.txt
Deleted hansard_2016-11-12_body.txt
Deleted hansard_2014-11-19_body.txt
Deleted hansard_2018-02-25_body.txt
Deleted hansard_2012-12-09_body.txt
Deleted hansard_2024-03-12_body.txt
Deleted hansard_2022-04-08_body.txt
Deleted hansard_2018-08-26_body.txt
Deleted hansard_2022-03-25_body.txt
Deleted hansard_2021-12-06_body.txt
Deleted hansard_2019-06-01_body.txt
Deleted hansard_2024-11-28_body.txt
Deleted hansard_2021-01-09_body.txt
Deleted hansard_2017-03-13_body.txt
Deleted hansard_2024-03-02_body.txt
Deleted hansard_2020-07-03_body.txt
Deleted hansard_2022-01-09_body.txt
Deleted hansard_2025-03-23_body.txt
Deleted hansard_2018-08-04_body.txt
Deleted hansard_2025-03-10_body.txt
Deleted hansard_2012-07-11_body.txt
Deleted hansard_2021-07-13_body.txt
Deleted hansard_2021-01-07_body.txt
Deleted hansard_2014-11-20_body.txt
Deleted hansard_2014-04-11_body.txt
Deleted hansard_2019-06-15_body.txt
Deleted hansard_2024-01-19_b

## Uploading the files to the Drive

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [40]:
!rm -f hansard_out.zip
!zip -r hansard_out.zip hansard_out/*

  adding: hansard_out/hansard_2012-02-07_body.html (deflated 71%)
  adding: hansard_out/hansard_2012-02-07_body.txt (deflated 67%)
  adding: hansard_out/hansard_2012-02-08_body.html (deflated 71%)
  adding: hansard_out/hansard_2012-02-08_body.txt (deflated 66%)
  adding: hansard_out/hansard_2012-02-09_body.html (deflated 71%)
  adding: hansard_out/hansard_2012-02-09_body.txt (deflated 67%)
  adding: hansard_out/hansard_2012-02-14_body.html (deflated 71%)
  adding: hansard_out/hansard_2012-02-14_body.txt (deflated 66%)
  adding: hansard_out/hansard_2012-02-15_body.html (deflated 71%)
  adding: hansard_out/hansard_2012-02-15_body.txt (deflated 66%)
  adding: hansard_out/hansard_2012-02-16_body.html (deflated 71%)
  adding: hansard_out/hansard_2012-02-16_body.txt (deflated 66%)
  adding: hansard_out/hansard_2012-02-28_body.html (deflated 72%)
  adding: hansard_out/hansard_2012-02-28_body.txt (deflated 68%)
  adding: hansard_out/hansard_2012-02-29_body.html (deflated 72%)
  adding: hansard

In [35]:
failed_dates = [
    # "2024-11-11", "2024-11-12", "2024-11-15", "2024-11-26", "2024-11-28",
    # "2024-11-29", "2024-11-30", "2024-12-07", "2024-12-08", "2024-12-10",
    # "2024-12-16", "2024-12-18", "2025-01-21", "2025-01-22", "2025-01-24",
    # "2025-02-25", "2025-02-27", "2025-03-06", "2025-03-07", "2025-03-08",
    # "2025-03-10", "2025-03-11", "2025-03-12", "2025-03-14", "2025-03-21",
    # "2025-03-22", "2025-03-23", "2025-03-24", "2025-04-15", "2025-04-16",
    # "2025-04-18", "2025-04-19", "2025-04-25", "2025-04-26", "2025-04-27",
    # "2025-04-28", "2025-04-29", "2025-05-25", "2025-05-26", "2025-05-27",
    # "2025-05-29", "2025-06-14",
#     "2012-06-03",
# "2012-06-04",
# "2012-06-05",
# "2012-06-29",
# "2012-07-07",
# "2012-07-09",
# "2012-07-11",
# "2012-08-01",
# "2012-08-02",
# "2012-08-03",
# "2012-08-11",
# "2012-08-12",
# "2012-08-14",
# "2012-08-15",
# "2012-08-26",
# "2012-08-28",
# "2012-08-29",
# "2012-09-18",
# "2012-09-30",
# "2012-10-01",
# "2012-10-02",
# "2012-10-03",
# "2012-10-04",
# "2012-11-04",
# "2012-11-06",
# "2012-11-07",
# "2012-11-08",
# "2012-11-14",
# "2012-11-15",
# "2012-11-16",
# "2012-12-09",
# "2012-12-11",
# "2012-12-12",
# "2013-04-04",
# "2013-05-14",
# "2013-05-15",
# "2013-05-16",
# "2013-07-08",
# "2013-07-09",
# "2013-07-10",
# "2013-07-11",
# "2013-07-17",
# "2013-07-18",
# "2013-07-19",
# "2013-07-21",
# "2013-08-14",
# "2013-08-15",
# "2013-08-18",
# "2013-08-20",
# "2013-09-06",
# "2013-09-07",
# "2013-09-08",
# "2013-09-16",
# "2013-09-18",
# "2013-09-19",
# "2013-12-10",
# "2013-12-11",
# "2013-12-12",
# "2013-12-13",
# "2014-03-19",
# "2014-04-11",
# "2014-04-18",
# "2014-04-19",
# "2014-04-20",
# "2014-04-22",
# "2014-05-13",
# "2014-05-14",
# "2014-05-15",
# "2014-05-16",
# "2014-05-17",
# "2014-07-28",
# "2014-08-06",
# "2014-08-07",
# "2014-08-08",
# "2014-08-10",
# "2014-08-18",
# "2014-08-19",
# "2014-08-20",
# "2014-10-17",
# "2014-10-18",
# "2014-10-20",
# "2014-10-21",
# "2014-10-23",
# "2014-11-09",
# "2014-11-10",
# "2014-11-11",
# "2014-11-19",
# "2014-11-20",
# "2014-11-21",
# "2014-11-22",
# "2014-11-29",
# "2014-11-30",
# "2014-12-02",
# "2014-12-03",
# "2014-12-10",
# "2015-01-19",
# "2015-02-24",
# "2015-02-25",
# "2015-02-26",
# "2015-03-19",
# "2015-03-21",
# "2015-03-22",
# "2015-03-23",
# "2015-04-17",
# "2015-04-23",
# "2015-04-25",
# "2015-04-26",
# "2015-05-08",
# "2015-05-09",
# "2015-05-11",
# "2015-05-12",
# "2015-06-02",
# "2015-06-03",
# "2015-06-04",
# "2015-06-06",
# "2015-06-28",
# "2015-06-29",
# "2015-06-30",
# "2015-07-01",
# "2015-07-07",
# "2015-07-08",
# "2015-07-09",
# "2015-07-10",
# "2015-07-11",
# "2015-07-17",
# "2015-07-18",
# "2015-07-19",
# "2015-07-20",
# "2015-07-21",
# "2015-07-29",
# "2015-08-13",
# "2015-08-15",
# "2015-08-26",
# "2015-08-27",
# "2015-08-28",
# "2015-08-29",
# "2015-08-31",
# "2015-09-01",
# "2015-09-02",
# "2015-09-03",
# "2015-09-04",
# "2015-09-06",
# "2015-09-07",
# "2015-09-08",
# "2015-09-09",
# "2015-09-16",
# "2015-09-30",
# "2015-10-01",
# "2015-10-02",
# "2015-10-03",
# "2015-10-04",
# "2015-11-05",
# "2015-11-06",
# "2015-11-08",
# "2015-12-10",
# "2015-12-11",
# "2015-12-29",
# "2015-12-30",
# "2015-12-31",
# "2016-01-01",
# "2016-01-08",
# "2016-01-09",
# "2016-01-13",
# "2016-01-15",
# "2016-01-16",
# "2016-01-17",
# "2016-02-02",
# "2016-02-03",
# "2016-02-04",
# "2016-02-05",
# "2016-02-06",
# "2016-02-17",
# "2016-02-18",
# "2016-02-19",
# "2016-02-21",
# "2016-05-12",
# "2016-05-14",
# "2016-07-12",
# "2016-07-13",
# "2016-07-14",
# "2016-07-15",
# "2016-08-25",
# "2016-08-29",
# "2016-09-15",
# "2016-09-16",
# "2016-09-17",
# "2016-10-14",
# "2016-10-15",
# "2016-10-16",
# "2016-10-17",
# "2016-10-18",
# "2016-11-08",
# "2016-11-09",
# "2016-11-12",
# "2016-12-03",
# "2016-12-05",
# "2016-12-06",
# "2016-12-30",
# "2016-12-31",
# "2017-01-08",
# "2017-01-10",
# "2017-01-11",
# "2017-01-19",
# "2017-01-20",
# "2017-01-21",
# "2017-01-28",
# "2017-01-29",
# "2017-01-30",
# "2017-01-31",
# "2017-02-06",
# "2017-02-07",
# "2017-02-08",
# "2017-02-16",
# "2017-02-17",
# "2017-02-18",
# "2017-02-19",
# "2017-02-20",
# "2017-03-07",
# "2017-03-13",
# "2017-03-14",
# "2017-03-15",
# "2017-03-16",
# "2017-03-17",
# "2017-03-28",
# "2017-03-29",
# "2017-03-30",
# "2017-04-27",
# "2017-04-28",
# "2017-04-29",
# "2017-04-30",
# "2017-05-02",
# "2017-05-05",
# "2017-05-12",
# "2017-05-13",
# "2017-05-14",
# "2017-05-15",
# "2017-05-27",
# "2017-05-28",
# "2017-05-31",
# "2017-06-08",
# "2017-06-09",
# "2017-06-10",
# "2017-06-17",
# "2017-07-01",
# "2017-07-02",
# "2017-07-03",
# "2017-07-05",
# "2017-07-11",
# "2017-07-12",
# "2017-07-13",
# "2017-07-14",
# "2017-07-15",
# "2017-08-31",
# "2017-09-01",
# "2017-09-02",
# "2017-09-03",
# "2017-09-11",
# "2017-10-25",
# "2017-11-03",
# "2017-11-04",
# "2017-11-05",
# "2017-11-06",
# "2017-11-07",
# "2017-11-28",
# "2017-11-30",
# "2017-12-01",
# "2017-12-02",
# "2018-01-03",
# "2018-01-05",
# "2018-01-17",
# "2018-01-19",
# "2018-01-21",
# "2018-02-07",
# "2018-02-08",
# "2018-02-09",
# "2018-02-16",
# "2018-02-17",
# "2018-02-19",
# "2018-02-20",
# "2018-02-21",
# "2018-02-25",
# "2018-03-03",
# "2018-03-05",
# "2018-03-07",
# "2018-03-17",
# "2018-04-07",
# "2018-04-08",
# "2018-04-09",
# "2018-04-10",
# "2018-05-02",
# "2018-05-04",
# "2018-05-05",
# "2018-05-06",
# "2018-05-12",
# "2018-05-13",
# "2018-05-14",
# "2018-05-15",
# "2018-05-16",
# "2018-06-26",
# "2018-06-27",
# "2018-06-30",
# "2018-07-06",
# "2018-07-31",
# "2018-08-01",
# "2018-08-02",
# "2018-08-03",
# "2018-08-04",
# "2018-08-25",
# "2018-08-26",
# "2018-08-27",
# "2018-08-28",
# "2018-10-15",
# "2018-10-16",
# "2018-10-17",
# "2018-10-18",
# "2018-10-24",
# "2018-10-25",
# "2018-10-26",
# "2018-10-27",
# "2018-10-28",
# "2018-10-29",
# "2018-10-30",
# "2018-10-31",
# "2018-11-01",
# "2018-11-02",
# "2018-11-07",
# "2018-11-29",
# "2018-11-30",
# "2018-12-01",
# "2018-12-02",
# "2018-12-14",
# "2019-01-02",
# "2019-01-04",
# "2019-01-05",
# "2019-02-06",
# "2019-02-09",
# "2019-02-11",
# "2019-02-12",
# "2019-02-13",
# "2019-02-14",
# "2019-02-15",
# "2019-03-13",
# "2019-03-14",
# "2019-03-15",
# "2019-03-16",
# "2019-03-18",
# "2019-03-19",
# "2019-03-20",
# "2019-03-21",
# "2019-04-13",
# "2019-04-14",
# "2019-04-18",
# "2019-04-19",
# "2019-04-20",
# "2019-05-22",
# "2019-05-23",
# "2019-05-24",
# "2019-05-25",
# "2019-05-26",
# "2019-06-01",
# "2019-06-03",
# "2019-06-04",
# "2019-06-12",
# "2019-06-13",
# "2019-06-14",
# "2019-06-15",
# "2019-07-06",
# "2019-07-07",
# "2019-07-08",
# "2019-07-10",
# "2019-08-11",
# "2019-08-14",
# "2019-09-14",
# "2019-09-15",
# "2019-09-16",
# "2019-09-17",
# "2019-09-18",
# "2019-09-24",
# "2019-09-26",
# "2019-09-27",
# "2019-09-28",
# "2019-10-05",
# "2019-10-06",
# "2019-10-08",
# "2019-11-18",
# "2019-11-19",
# "2019-11-20",
# "2019-11-21",
# "2019-11-22",
# "2019-12-16",
# "2019-12-17",
# "2020-01-07",
# "2020-01-08",
# "2020-01-09",
# "2020-01-17",
# "2020-01-18",
# "2020-01-19",
# "2020-01-20",
# "2020-01-21",
# "2020-03-02",
# "2020-03-03",
# "2020-03-04",
# "2020-03-06",
# "2020-03-17",
# "2020-03-18",
# "2020-03-27",
# "2020-03-29",
# "2020-03-30",
# "2020-03-31",
# "2020-04-06",
# "2020-04-09",
# "2020-04-10",
# "2020-04-16",
# "2020-04-17",
# "2020-04-18",
# "2020-04-19",
# "2020-05-25",
# "2020-06-05",
# "2020-06-07",
# "2020-06-09",
# "2020-06-30",
# "2020-07-01",
# "2020-07-02",
# "2020-07-03",
# "2020-07-04",
# "2020-07-06",
# "2020-07-07",
# "2020-07-08",
# "2020-07-11",
# "2020-07-12",
# "2020-07-13",
# "2020-07-14",
# "2020-09-06",
# "2020-09-07",
# "2020-10-10",
# "2020-10-28",
# "2020-10-29",
# "2020-10-30",
# "2020-10-31",
# "2020-11-01",
# "2020-11-02",
# "2020-11-03",
# "2020-11-04",
# "2020-11-05",
# "2020-11-06",
# "2020-11-12",
# "2020-11-14",
# "2020-11-15",
# "2020-11-16",
# "2020-11-23",
# "2020-11-24",
# "2020-11-25",
# "2020-11-26",
# "2020-12-03",
# "2020-12-17",
# "2020-12-21",
# "2020-12-28",
# "2020-12-29",
# "2021-01-06",
# "2021-01-07",
# "2021-01-09",
# "2021-01-26",
# "2021-01-29",
# "2021-01-30",
# "2021-02-20",
# "2021-02-21",
# "2021-02-23",
# "2021-02-24",
# "2021-03-03",
# "2021-03-17",
# "2021-03-18",
# "2021-03-19",
# "2021-03-20",
# "2021-03-21",
# "2021-04-12",
# "2021-04-13",
# "2021-04-15",
# "2021-05-06",
# "2021-05-08",
# "2021-05-09",
# "2021-05-21",
# "2021-05-22",
# "2021-05-24",
# "2021-05-25",
# "2021-06-01",
# "2021-06-02",
# "2021-06-04",
# "2021-06-13",
# "2021-06-20",
# "2021-06-21",
# "2021-06-22",
# "2021-06-23",
# "2021-06-24",
# "2021-07-05",
# "2021-07-06",
# "2021-07-07",
# "2021-07-09",
# "2021-07-10",
# "2021-07-13",
# "2021-07-21",
# "2021-07-22",
# "2021-07-23",
# "2021-09-14",
# "2021-09-15",
# "2021-09-16",
# "2021-09-17",
# "2021-09-24",
# "2021-10-08",
# "2021-10-09",
# "2021-10-10",
# "2021-10-11",
# "2021-10-12",
# "2021-10-18",
# "2021-10-19",
# "2021-10-20",
# "2021-10-21",
# "2021-10-22",
# "2021-10-28",
# "2021-10-29",
# "2021-10-30",
# "2021-10-31",
# "2021-11-01",
# "2021-11-08",
# "2021-11-09",
# "2021-11-23",
# "2021-11-24",
# "2021-12-02",
# "2021-12-03",
# "2021-12-05",
# "2021-12-06",
# "2021-12-13",
# "2021-12-15",
# "2021-12-16",
# "2021-12-28",
# "2021-12-30",
# "2022-01-06",
# "2022-01-07",
# "2022-01-08",
# "2022-01-09",
# "2022-01-10",
# "2022-01-26",
# "2022-01-28",
# "2022-01-29",
# "2022-01-30",
# "2022-03-23",
# "2022-03-25",
# "2022-03-27",
# "2022-03-28",
# "2022-03-29",
# "2022-03-30",
# "2022-03-31",
# "2022-04-06",
# "2022-04-07",
# "2022-04-08",
# "2022-04-09",
# "2022-04-10",
# "2022-04-26",
# "2022-04-27",
# "2022-04-28",
# "2022-04-29",
# "2022-04-30",
# "2022-05-11",
# "2022-05-12",
# "2022-05-13",
# "2022-05-31",
# "2022-06-01",
# "2022-06-02",
# "2022-06-03",
# "2022-06-04",
# "2022-06-25",
# "2022-06-26",
# "2022-06-27",
# "2022-06-28",
# "2022-06-29",
# "2022-07-30",
# "2022-07-31",
# "2022-08-02",
# "2022-09-03",
# "2022-09-04",
# "2022-09-05",
# "2022-09-06",
# "2022-09-07",
# "2022-09-24",
# "2022-09-25",
# "2022-09-26",
# "2022-09-27",
# "2022-10-18",
# "2022-10-19",
# "2022-10-20",
# "2022-10-22",
# "2022-11-06",
# "2023-01-26",
# "2023-01-27",
# "2023-01-28",
# "2023-01-29",
# "2023-01-30",
# "2023-02-25",
# "2023-02-26",
# "2023-02-27",
# "2023-02-28",
# "2023-03-01",
# "2023-04-06",
# "2023-04-10",
# "2023-04-17",
# "2023-04-19",
# "2023-04-20",
# "2023-05-26",
# "2023-05-27",
# "2023-05-30",
# "2023-06-10",
# "2023-06-12",
# "2023-06-13",
# "2023-06-14",
# "2023-06-25",
# "2023-06-26",
# "2023-06-27",
# "2023-06-28",
# "2023-06-29",
# "2023-08-14",
# "2023-08-15",
# "2023-08-16",
# "2023-08-17",
# "2023-08-18",
# "2023-08-29",
# "2023-08-30",
# "2023-08-31",
# "2023-09-01",
# "2023-09-02",
# "2023-09-14",
# "2023-09-25",
# "2023-09-26",
# "2023-09-27",
# "2023-11-07",
# "2023-11-08",
# "2023-11-09",
# "2023-11-11",
# "2023-12-07",
# "2023-12-08",
# "2023-12-09",
# "2023-12-10",
# "2023-12-11",
# "2023-12-12",
# "2023-12-14",
# "2023-12-15",
# "2023-12-16",
# "2023-12-23",
# "2023-12-24",
# "2023-12-26",
# "2024-01-16",
# "2024-01-17",
# "2024-01-18",
# "2024-01-19",
# "2024-01-20",
# "2024-02-08",
# "2024-02-09",
# "2024-03-01",
# "2024-03-02",
# "2024-03-04",
# "2024-03-05",
# "2024-03-12",
# "2024-03-15",
# "2024-04-15",
# "2024-04-16",
# "2024-04-17",
# "2024-04-18",
# "2024-05-11",
# "2024-05-12",
# "2024-05-13",
# "2024-05-14",
# "2024-06-15",
# "2024-06-16",
# "2024-06-17",
# "2024-06-18",
# "2024-06-25",
# "2024-06-27",
# "2024-07-09",
# "2024-07-11",
# "2024-07-12",
# "2024-07-13",
# "2024-07-14",
# "2024-07-15",
# "2024-07-16",
# "2024-07-17",
# "2024-07-20",
# "2024-07-29",
# "2024-07-30",
# "2024-07-31",
# "2024-08-01",
# "2024-08-02",
# "2024-08-14",
# "2024-08-15",
# "2024-08-17",
# "2024-08-23",
# "2024-08-25",
# "2024-08-27",
# "2024-09-02",
# "2024-09-03",
# "2024-09-17",
# "2024-09-18",
# "2024-09-19",
# "2024-09-20",
# "2024-09-21",
# "2024-10-07",
# "2024-10-10",
# "2024-10-11",
# "2024-10-22",
# "2024-10-23",
# "2024-10-24",
# "2024-10-25",
# "2024-10-26",
# "2024-11-01",
# "2024-11-02",
# "2024-11-03",
# "2024-11-05",
    "2015-06-28",
"2015-06-29",
"2015-06-30",
"2015-07-01",
"2015-07-07",
"2015-07-21",
"2015-07-29",
"2015-10-04",
"2015-11-05",
"2015-11-06",
"2015-11-08",
"2015-12-10",
"2016-01-09",
"2016-01-13",
"2016-02-06",
"2016-02-17",
"2016-02-18",
"2016-02-19",
"2017-01-28",
"2017-01-29",
"2017-01-30",
"2017-01-31",
    "2017-02-06",
"2017-05-02",
"2017-05-12",
"2017-05-14",
"2017-07-15",
"2017-11-03",
"2017-11-04",
"2017-12-01",
"2018-01-03",
"2018-01-05",
"2018-02-17",
"2018-02-20",
"2018-02-21",
"2018-04-07",
"2018-05-02",
"2018-05-04",
"2018-05-06",
"2018-07-31",
"2018-08-01",
"2018-08-02",
"2018-10-17",
"2019-03-14",
"2019-03-16",
"2019-04-14",
"2019-04-18",
"2019-04-20",
"2019-05-22",
"2019-08-14",
"2019-09-16",
"2019-09-17",
"2019-09-18",
"2019-09-24",
"2020-03-17",
"2020-03-18",
"2020-10-30",
"2019-03-13",
"2019-09-26",
"2020-06-30",
"2015-06-28",
"2015-06-29",
    "2021-07-07",
    "2024-10-23"
]

In [36]:
import asyncio
from typing import Dict, Any, List

async def scrape_specific_dates(date_list: List[str], out_dir: str = ".", save: bool = True, batch_size: int = 5, max_retries: int = 3):
    """
    Loops through a specific list of dates and calls get_hansard_body().
    Downloads in batches concurrently using asyncio.gather.
    If a date fails, logs and continues. Retries specific failures.
    Returns:
      - results: list of successful result dicts
      - failures: list of {"date":..., "error":...}
    """
    results: List[Dict[str, Any]] = []
    initial_failures: List[Dict[str, str]] = []
    retry_queue: List[Dict[str, Any]] = [] # [{'date': 'YYYY-MM-DD', 'attempt': 0}]

    # Initial pass
    for i in range(0, len(date_list), batch_size):
        batch_dates = date_list[i : i + batch_size]
        tasks = []
        for d in batch_dates:
            tasks.append(get_hansard_body(d, save=save, out_dir=out_dir))

        batch_results = await asyncio.gather(*tasks, return_exceptions=True)

        for d, res in zip(batch_dates, batch_results):
            if isinstance(res, Exception):
                error_msg = str(res)
                if "Blocked by anti-bot verification page." in error_msg and max_retries > 0:
                    retry_queue.append({'date': d, 'attempt': 0})
                    print(f"⚠️ {d} BLOCKED, adding to retry queue.")
                else:
                    initial_failures.append({"date": d, "error": error_msg})
                    print(f"⚠️ {d} FAILED -> {error_msg}")
            else:
                results.append(res)
                print(f"✅ {d} OK  -> {res.get('saved_txt_path')}")

        await asyncio.sleep(1) # Polite delay between batches

    # Retry pass
    while retry_queue:
        current_retry_batch = []
        next_retry_queue = []

        # Prepare a batch for retrying
        for item in retry_queue:
            if item['attempt'] < max_retries:
                current_retry_batch.append(item)
            else:
                initial_failures.append({"date": item['date'], "error": f"Failed after {max_retries} retries: Blocked by anti-bot verification page."})

        if not current_retry_batch: # No more items to retry
            break

        print(f"\n🔄 Retrying {len(current_retry_batch)} dates (attempt {current_retry_batch[0]['attempt'] + 1}/{max_retries})...")
        retry_tasks = []
        retry_dates_map = {item['date']: item for item in current_retry_batch}

        for item in current_retry_batch:
            retry_tasks.append(get_hansard_body(item['date'], save=save, out_dir=out_dir))

        retry_batch_results = await asyncio.gather(*retry_tasks, return_exceptions=True)

        for date_item, res in zip(current_retry_batch, retry_batch_results):
            d = date_item['date']
            if isinstance(res, Exception):
                error_msg = str(res)
                if "Blocked by anti-bot verification page." in error_msg:
                    date_item['attempt'] += 1
                    if date_item['attempt'] < max_retries:
                        next_retry_queue.append(date_item)
                        print(f"⚠️ {d} BLOCKED again, re-adding to retry queue (attempt {date_item['attempt'] + 1}).")
                    else:
                        initial_failures.append({"date": d, "error": f"Failed after {max_retries} retries: Blocked by anti-bot verification page."})
                        print(f"❌ {d} FAILED permanently after {max_retries} retries.")
                else:
                    initial_failures.append({"date": d, "error": error_msg})
                    print(f"⚠️ {d} FAILED during retry -> {error_msg}")
            else:
                results.append(res)
                print(f"✅ {d} OK on retry -> {res.get('saved_txt_path')}")

        retry_queue = next_retry_queue
        if retry_queue: # Only sleep if there are still items to retry
            await asyncio.sleep(5) # Longer polite delay between retry batches

    return results, initial_failures

In [37]:
retry_results, retry_failures = await scrape_specific_dates(failed_dates, out_dir="hansard_out", save=True, batch_size=4, max_retries=3)

print("Retry Success:", len(retry_results))
print("Retry Failures:", len(retry_failures))
retry_failures[:5]

✅ 2015-06-28 OK  -> hansard_out/hansard_2015-06-28_body.txt
⚠️ 2015-06-29 BLOCKED, adding to retry queue.
✅ 2015-06-30 OK  -> hansard_out/hansard_2015-06-30_body.txt
⚠️ 2015-07-01 BLOCKED, adding to retry queue.
✅ 2015-07-07 OK  -> hansard_out/hansard_2015-07-07_body.txt
✅ 2015-07-21 OK  -> hansard_out/hansard_2015-07-21_body.txt
✅ 2015-07-29 OK  -> hansard_out/hansard_2015-07-29_body.txt
✅ 2015-10-04 OK  -> hansard_out/hansard_2015-10-04_body.txt
✅ 2015-11-05 OK  -> hansard_out/hansard_2015-11-05_body.txt
✅ 2015-11-06 OK  -> hansard_out/hansard_2015-11-06_body.txt
✅ 2015-11-08 OK  -> hansard_out/hansard_2015-11-08_body.txt
✅ 2015-12-10 OK  -> hansard_out/hansard_2015-12-10_body.txt
✅ 2016-01-09 OK  -> hansard_out/hansard_2016-01-09_body.txt
✅ 2016-01-13 OK  -> hansard_out/hansard_2016-01-13_body.txt
✅ 2016-02-06 OK  -> hansard_out/hansard_2016-02-06_body.txt
✅ 2016-02-17 OK  -> hansard_out/hansard_2016-02-17_body.txt
✅ 2016-02-18 OK  -> hansard_out/hansard_2016-02-18_body.txt
✅ 2016-0

[]

# move files

In [41]:
!rm -rf "/content/drive/MyDrive/NZ/University of Waikato Māori Data Findability Project/downloaded/hansard_out.zip"
!cp -r hansard_out.zip "/content/drive/MyDrive/NZ/University of Waikato Māori Data Findability Project/downloaded/hansard_out.zip"

# Download latest files from the drive and extract
### (continue from here if you've previously uploaded the dataset to the drive already)

In [43]:
!rm -rf "/content/hansard_to_process.zip"
!rm -rf "/content/hansard_to_process"
!cp -r "/content/drive/MyDrive/NZ/University of Waikato Māori Data Findability Project/downloaded/hansard_out.zip" "/content/hansard_to_process.zip"

In [44]:
!unzip -q hansard_to_process.zip -d hansard_to_process

# count # of .html files

In [45]:
!find "/content/hansard_to_process/hansard_out" -type f -name "*.html" | wc -l

1093


In [46]:
!find "/content/hansard_to_process/hansard_out" -type f -name "*.txt" | wc -l

1093


# Check wheter there are Maori words

In [47]:
!grep -R -l -i -E "māori|maori|iwi|hapū|whānau|whanau|te reo|tikanga|rangatiratanga|kaitiakitanga" \
"/content/hansard_to_process/hansard_out"

/content/hansard_to_process/hansard_out/hansard_2021-04-13_body.txt
/content/hansard_to_process/hansard_out/hansard_2022-05-12_body.html
/content/hansard_to_process/hansard_out/hansard_2015-10-13_body.html
/content/hansard_to_process/hansard_out/hansard_2013-10-16_body.txt
/content/hansard_to_process/hansard_out/hansard_2013-08-20_body.txt
/content/hansard_to_process/hansard_out/hansard_2020-09-02_body.html
/content/hansard_to_process/hansard_out/hansard_2018-06-12_body.html
/content/hansard_to_process/hansard_out/hansard_2018-09-19_body.html
/content/hansard_to_process/hansard_out/hansard_2017-12-20_body.txt
/content/hansard_to_process/hansard_out/hansard_2019-08-28_body.txt
/content/hansard_to_process/hansard_out/hansard_2014-05-29_body.html
/content/hansard_to_process/hansard_out/hansard_2019-11-12_body.html
/content/hansard_to_process/hansard_out/hansard_2014-01-29_body.html
/content/hansard_to_process/hansard_out/hansard_2021-02-09_body.html
/content/hansard_to_process/hansard_out

# Next steps

1. prep a dataset (orgin maori, not maori)

prep at lean 1000 data points

2. annotate it using gemini
3. split to train and test and fine tune LLM using the training test
4. evaluate it on the testset and basline modele (xGboost or something)


which is good?